In [1]:

import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "peft", "bitsandbytes", "accelerate"], check=True)


from pathlib import Path
import torch, json, gc
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT      = Path('/content/drive/MyDrive')
BASE_MODEL_PATH = DRIVE_ROOT / 'models' / 'Qwen2.5-32B-Instruct'
LORA_PATH       = DRIVE_ROOT / 'models' / 'cbt-qwen32b-lora'
RAG_DIR         = DRIVE_ROOT / 'cbt_rag'

EMBED_MODEL_ID    = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
RAG_DEVICE        = "cpu"


print("Loading corpus...")
with open(RAG_DIR / "corpus_with_ids.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)
corpus_embeddings = np.load(RAG_DIR / "corpus_embeddings.npy")
corpus_by_id = {c["id"]: c for c in corpus}
print(f"Corpus loaded: {len(corpus)} chunks")

print("Loading embedding model...")
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
embed_model     = AutoModel.from_pretrained(EMBED_MODEL_ID, torch_dtype=torch.float32).to(RAG_DEVICE)
embed_model.eval()

print("Loading reranker model...")
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_ID, padding_side='left')
reranker_lm        = AutoModelForCausalLM.from_pretrained(RERANKER_MODEL_ID, torch_dtype=torch.float32).to(RAG_DEVICE)
reranker_lm.eval()
print("RAG pipeline ready.")


def embed_texts(texts, batch_size=16, max_length=512):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = embed_tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(RAG_DEVICE)
        with torch.no_grad():
            outputs     = embed_model(**inputs)
            last_hidden = outputs.last_hidden_state
            attn_mask   = inputs["attention_mask"]
            seq_lens    = attn_mask.sum(dim=1) - 1
            batch_idx   = torch.arange(last_hidden.shape[0], device=RAG_DEVICE)
            pooled      = last_hidden[batch_idx, seq_lens]
            pooled      = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeds.append(pooled.float().cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

def retrieve(query, top_k=10):
    query_emb = embed_texts([query])
    sims      = (corpus_embeddings @ query_emb.T).squeeze(-1)
    top_idx   = np.argsort(-sims)[:top_k]
    return [{**corpus[idx], "similarity": float(sims[idx])} for idx in top_idx]

RERANK_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query "
    "and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n"
    "<|im_start|>user\n"
)
RERANK_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
RERANK_INSTRUCTION = (
    "Given a conversation excerpt from a CBT therapy session, retrieve relevant "
    "dialogue examples, transition rules, technique guidance, or safety information "
    "that would help a therapist respond appropriately to the next turn."
)
YES_TOKEN = reranker_tokenizer.convert_tokens_to_ids("yes")
NO_TOKEN  = reranker_tokenizer.convert_tokens_to_ids("no")

def rerank(query, candidates, top_k=5, batch_size=8, max_length=1024):
    scored = []
    for i in range(0, len(candidates), batch_size):
        batch   = candidates[i:i+batch_size]
        prompts = [
            f"{RERANK_PREFIX}<Instruct>: {RERANK_INSTRUCTION}\n"
            f"<Query>: {query}\n<Document>: {c['content'][:400]}{RERANK_SUFFIX}"
            for c in batch
        ]
        inputs = reranker_tokenizer(
            prompts, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(RAG_DEVICE)
        with torch.no_grad():
            logits     = reranker_lm(**inputs).logits[:, -1, :]
            yes_logits = logits[:, YES_TOKEN]
            no_logits  = logits[:, NO_TOKEN]
            stacked    = torch.stack([no_logits, yes_logits], dim=1)
            probs      = torch.softmax(stacked, dim=1)[:, 1]
        for c, p in zip(batch, probs.float().cpu().tolist()):
            scored.append({**c, "rerank_score": p})
    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]

SAFETY_CHUNKS = [c for c in corpus if c["layer"] == "Safety_fallback"]

def retrieve_and_rerank(query, retrieve_k=15, final_k=5, always_include_safety=True):
    candidates = retrieve(query, top_k=retrieve_k)
    if always_include_safety:
        existing_ids = set(c["id"] for c in candidates)
        for s in SAFETY_CHUNKS:
            if s["id"] not in existing_ids:
                candidates.append({**s, "similarity": None})
    return rerank(query, candidates, top_k=final_k)

def format_rag_context(results, max_chars=1500):
    lines      = ["[Retrieved context — for reference, not to be quoted directly]"]
    used_chars = 0
    for r in results:
        block = "\n(" + r["layer"] + ") " + r["content"]
        if used_chars + len(block) > max_chars:
            break
        lines.append(block)
        used_chars += len(block)
    return "\n".join(lines)


embed_model.to("cpu")
reranker_lm.to("cpu")
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading Base model...")
_base = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH), quantization_config=bnb, device_map="auto"
)
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(_base, str(LORA_PATH))
model.eval()
tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH))
print(f"FT+RAG model ready. GPU: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


THERAPIST_SYSTEM = (
    "You are a compassionate and skilled CBT (Cognitive Behavioral Therapy) "
    "therapist. You help clients identify, examine, and reframe unhelpful "
    "thinking patterns using evidence-based techniques including Socratic "
    "questioning, thought records, the cognitive model, and problem-solving. "
    "You are warm, non-judgmental, and clinically precise. "
    "You never provide diagnoses or replace professional care."
)

def _input_device(m):
    return next(m.parameters()).device


import html
import ipywidgets as widgets
from IPython.display import display, HTML

APP_WIDTH        = 900
messages_history = [{"role": "system", "content": THERAPIST_SYSTEM}]

display(HTML('''
<style>
.cbt-chat-app, .cbt-chat-app * {
  box-sizing: border-box;
  font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
}
.cbt-chat-app textarea {
  background: #fffdf9 !important;
  color: #22312a !important;
  border: 1px solid #ddd3c6 !important;
  border-radius: 16px !important;
  padding: 15px 17px !important;
  font-size: 15px !important;
  line-height: 1.45 !important;
  outline: none !important;
  resize: none !important;
  box-shadow: inset 0 1px 2px rgba(38,48,42,.04), 0 1px 0 rgba(255,255,255,.7) !important;
}
.cbt-chat-app textarea:focus {
  border-color: #86a996 !important;
  box-shadow: 0 0 0 4px rgba(134,169,150,.18), inset 0 1px 2px rgba(38,48,42,.04) !important;
}
.cbt-chat-app textarea::placeholder {
  color: #8a958e !important;
  opacity: 1 !important;
}
.cbt-chat-app .widget-button button {
  height: 52px !important;
  border: 0 !important;
  border-radius: 16px !important;
  background: #2f4a3d !important;
  color: #fff !important;
  font-size: 15px !important;
  font-weight: 800 !important;
  box-shadow: 0 12px 24px rgba(47,74,61,.22) !important;
  transition: transform .12s ease, background .12s ease, box-shadow .12s ease !important;
}
.cbt-chat-app .widget-button button:hover {
  background: #263d32 !important;
  transform: translateY(-1px);
  box-shadow: 0 16px 28px rgba(47,74,61,.26) !important;
}
.cbt-chat-app .widget-button button:disabled {
  opacity: .65 !important;
  transform: none !important;
}
.cbt-chat-app .reset-btn button {
  background: #f0f4f1 !important;
  color: #2f4a3d !important;
  font-size: 13px !important;
  font-weight: 700 !important;
  box-shadow: none !important;
  border: 1px solid #c8d8cc !important;
}
.cbt-chat-app .reset-btn button:hover {
  background: #e2ece5 !important;
  transform: none !important;
  box-shadow: none !important;
}
</style>
'''))

def make_bubble(text, is_user):
    safe_text = html.escape(text).replace('\n', '<br>')
    if is_user:
        return f'''
        <div style="display:flex;justify-content:flex-end;align-items:flex-end;margin:14px 0;">
          <div style="max-width:70%;background:#2f4a3d;color:#fff;padding:13px 16px;border-radius:18px 18px 6px 18px;font-size:15px;line-height:1.6;box-shadow:0 12px 26px rgba(47,74,61,.18);word-break:break-word;">{safe_text}</div>
          <div style="width:34px;height:34px;background:#dcebe1;color:#2f4a3d;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:900;flex-shrink:0;margin-left:10px;">You</div>
        </div>'''
    return f'''
    <div style="display:flex;justify-content:flex-start;align-items:flex-end;margin:14px 0;">
      <div style="width:34px;height:34px;background:#dcebe1;color:#2f4a3d;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:900;flex-shrink:0;margin-right:10px;">CBT</div>
      <div style="max-width:70%;background:#fff;color:#22312a;padding:13px 16px;border:1px solid #e4dbd0;border-radius:18px 18px 18px 6px;font-size:15px;line-height:1.6;box-shadow:0 12px 26px rgba(32,43,36,.10);word-break:break-word;">{safe_text}</div>
    </div>'''

initial_msg = make_bubble("Hello, I'm here to support you. What's been on your mind lately?", False)
msgs_html   = [initial_msg]

def render_panel(messages_html, status='Ready'):
    status_bg = '#eef7f1' if status == 'Ready' else '#fff3d6' if status == 'Typing' else '#ffe8e0'
    status_fg = '#2f4a3d' if status == 'Ready' else '#745b16' if status == 'Typing' else '#8b2d20'
    return f'''
<div style="width:{APP_WIDTH}px;margin:0 auto;border:1px solid #d8d0c6;border-radius:20px 20px 0 0;overflow:hidden;background:#fbfaf7;box-shadow:0 24px 56px rgba(22,31,26,.22);">
  <div style="height:96px;background:#2f4a3d;padding:0 28px;display:flex;align-items:center;justify-content:space-between;">
    <div style="display:flex;align-items:center;gap:15px;">
      <div style="width:52px;height:52px;background:#dcebe1;color:#2f4a3d;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:14px;font-weight:900;">CBT</div>
      <div>
        <div style="color:#fff;font-size:24px;font-weight:850;letter-spacing:0;">Fine-tuned + RAG Qwen2.5-32B CBT Model</div>
        <div style="color:#bdd8c9;font-size:13px;margin-top:5px;">Powered by Qwen2.5-32B · Fine-tuned + RAG</div>
      </div>
    </div>
    <div style="background:{status_bg};color:{status_fg};border-radius:999px;padding:8px 14px;font-size:13px;font-weight:800;">{status}</div>
  </div>
  <div style="background:linear-gradient(180deg,#fffdf9 0%,#f6f1e9 100%);height:500px;overflow-y:auto;padding:26px 30px;">
    {''.join(messages_html)}
  </div>
</div>
'''

def refresh_chat(status='Ready'):
    chat_output.value = render_panel(msgs_html, status=status)

chat_output = widgets.HTML(value=render_panel(msgs_html))

text_input = widgets.Textarea(
    placeholder="Share what's on your mind...",
    layout=widgets.Layout(width=f'{APP_WIDTH - 164}px', height='66px')
)
send_btn = widgets.Button(
    description='Send',
    layout=widgets.Layout(width='112px', height='52px')
)
reset_btn = widgets.Button(
    description='New Conversation',
    layout=widgets.Layout(width=f'{APP_WIDTH}px', height='44px', margin='0 auto')
)
reset_btn.add_class('reset-btn')

composer = widgets.HBox(
    [text_input, send_btn],
    layout=widgets.Layout(
        width=f'{APP_WIDTH}px',
        margin='0 auto',
        padding='14px 18px 16px',
        background='#fbfaf7',
        border='1px solid #d8d0c6',
        border_top='1px solid #e7dfd5',
        border_radius='0 0 0 0',
        align_items='center',
        justify_content='space-between'
    )
)

reset_bar = widgets.HBox(
    [reset_btn],
    layout=widgets.Layout(
        width=f'{APP_WIDTH}px',
        margin='0 auto',
        padding='0 18px 14px',
        background='#fbfaf7',
        border='1px solid #d8d0c6',
        border_top='none',
        border_radius='0 0 20px 20px',
        justify_content='center'
    )
)

root = widgets.VBox(
    [chat_output, composer, reset_bar],
    layout=widgets.Layout(width=f'{APP_WIDTH}px', margin='0 auto')
)
root.add_class('cbt-chat-app')

def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value    = ''
    send_btn.disabled   = True
    text_input.disabled = True

    msgs_html.append(make_bubble(user_msg, True))
    msgs_html.append(make_bubble('Typing...', False))
    refresh_chat(status='Typing')

    try:

        results          = retrieve_and_rerank(user_msg, retrieve_k=15, final_k=5)
        rag_context      = format_rag_context(results)
        augmented_system = THERAPIST_SYSTEM + "\n\n" + rag_context


        rag_messages = [{"role": "system", "content": augmented_system}]
        for m in messages_history[1:]:
            rag_messages.append(m)
        rag_messages.append({"role": "user", "content": user_msg})

        prompt_text = tokenizer.apply_chat_template(
            rag_messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=False)
        device = _input_device(model)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()


        messages_history.append({"role": "user",      "content": user_msg})
        messages_history.append({"role": "assistant", "content": response})

        msgs_html.pop()
        msgs_html.append(make_bubble(response or '(empty response)', False))
        refresh_chat(status='Ready')

    except Exception as exc:
        msgs_html.pop()
        msgs_html.append(make_bubble(f'Generation failed: {type(exc).__name__}: {exc}', False))
        refresh_chat(status='Error')
        raise
    finally:
        send_btn.disabled   = False
        text_input.disabled = False

def on_reset(b):
    global msgs_html, messages_history
    messages_history = [{"role": "system", "content": THERAPIST_SYSTEM}]
    msgs_html = [make_bubble("Hello, I'm here to support you. What's been on your mind lately?", False)]
    refresh_chat(status='Ready')

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)
display(root)

Mounted at /content/drive
Loading corpus...
Corpus loaded: 217 chunks
Loading embedding model...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

RAG pipeline ready.
Free VRAM: 39.1 GB
Loading Base model...


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import json
import pandas as pd
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE   = Path('/content/drive/MyDrive')
MT_DIR  = DRIVE / 'cbt_multiturn_results'
RAG_DIR = DRIVE / 'cbt_rag_results'
OUT_DIR = DRIVE / 'CBT' / '6.25 final results'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

base_dialogues   = load_json(MT_DIR  / 'base_dialogues.json')
ft_dialogues     = load_json(MT_DIR  / 'ft_dialogues.json')
ft_rag_dialogues = load_json(MT_DIR  / 'ft_rag_dialogues.json')
rag_dialogues    = load_json(RAG_DIR / 'rag_dialogues.json')


SCENARIOS = [
    {
        "id": 1, "category": "exam_anxiety",
        "profile": "A university student with intense exam anxiety and fear of failure.",
        "initial_patient_message": "I feel extremely anxious about my exams. I keep imagining that if I fail, my future is over.",
        "core_beliefs": ["My worth depends on achievement", "Failure would be catastrophic"],
        "target_dynamics": "The patient starts anxious, reveals catastrophizing, then avoidance of studying, and may accept a small behavioral step if guided well.",
    },
    {
        "id": 2, "category": "perfectionism",
        "profile": "A postgraduate student who repeatedly rewrites assignments and cannot tolerate imperfect work.",
        "initial_patient_message": "I have rewritten the same section of my dissertation for days. It never feels good enough.",
        "core_beliefs": ["If it is not perfect, it is worthless", "Mistakes mean I am inadequate"],
        "target_dynamics": "The patient defends high standards at first, then reveals exhaustion and fear of judgment.",
    },
    {
        "id": 3, "category": "social_anxiety",
        "profile": "A student afraid to speak in seminars because they believe others will judge them.",
        "initial_patient_message": "I want to contribute in class, but I freeze because I am sure everyone will think I sound stupid.",
        "core_beliefs": ["People are evaluating me harshly", "If I sound uncertain, I will be rejected"],
        "target_dynamics": "The patient worries about embarrassment and needs help testing mind-reading assumptions.",
    },
    {
        "id": 4, "category": "avoidance",
        "profile": "A student avoiding a major assignment because starting triggers shame and anxiety.",
        "initial_patient_message": "I keep avoiding my assignment. Even opening the document makes me feel sick.",
        "core_beliefs": ["If I start, I will prove I cannot do it", "Avoidance is safer than trying"],
        "target_dynamics": "The patient describes avoidance relief, then the long-term cost, and may consider a tiny first step.",
    },
    {
        "id": 5, "category": "negative_self_talk",
        "profile": "A student with harsh internal criticism after receiving feedback.",
        "initial_patient_message": "My tutor gave me critical feedback and now I keep thinking I am just not smart enough.",
        "core_beliefs": ["Criticism means I am not capable", "I should already know how to do this"],
        "target_dynamics": "The patient initially overgeneralizes feedback and needs help separating facts from self-attack.",
    },
    {
        "id": 6, "category": "burnout",
        "profile": "A high-achieving student who is exhausted but feels guilty resting.",
        "initial_patient_message": "I am exhausted all the time, but when I rest I feel lazy and guilty.",
        "core_beliefs": ["Rest must be earned", "If I stop working, I will fall behind"],
        "target_dynamics": "The patient is depleted, may resist rest, and benefits from reframing rest as part of functioning.",
    },
    {
        "id": 7, "category": "reassurance_seeking",
        "profile": "A student who repeatedly asks classmates whether their work is good enough.",
        "initial_patient_message": "I keep asking people if my essay is okay, but even when they reassure me I still feel uncertain.",
        "core_beliefs": ["I cannot trust my own judgment", "Uncertainty is dangerous"],
        "target_dynamics": "The patient describes temporary relief, repeated checking, and fear of submitting independently.",
    },
    {
        "id": 8, "category": "future_worry",
        "profile": "A final-year student worried about employment and life after graduation.",
        "initial_patient_message": "I cannot stop worrying about graduation. What if I never find a job and fall behind everyone else?",
        "core_beliefs": ["My future is fragile", "I must solve everything now"],
        "target_dynamics": "The patient jumps far into the future and needs grounding in controllable next steps.",
    },
    {
        "id": 9, "category": "identity",
        "profile": "A student whose identity is strongly tied to academic success.",
        "initial_patient_message": "If I am not doing well academically, I honestly do not know who I am.",
        "core_beliefs": ["Achievement is my identity", "Without success, I have no value"],
        "target_dynamics": "The patient explores worth, roles outside academics, and fear of losing status.",
    },
    {
        "id": 10, "category": "sleep_stress",
        "profile": "A student with racing thoughts at night about deadlines and performance.",
        "initial_patient_message": "At night my mind races through everything I have not done, and then I cannot sleep.",
        "core_beliefs": ["I must stay alert to stay safe", "If I rest, things will fall apart"],
        "target_dynamics": "The patient describes nighttime rumination and benefits from worry scheduling and realistic planning.",
    },
    {
        "id": 11, "category": "hopelessness",
        "profile": "A student who feels discouraged after repeated academic setbacks, without imminent self-harm intent.",
        "initial_patient_message": "I keep trying and still fall behind. I feel like nothing I do actually changes anything.",
        "core_beliefs": ["Effort does not matter", "I am permanently stuck"],
        "target_dynamics": "The patient is discouraged and needs validation, evidence review, and very small signs of agency.",
    },
    {
        "id": 12, "category": "self_compassion",
        "profile": "A student who believes self-kindness will reduce motivation.",
        "initial_patient_message": "People tell me to be kinder to myself, but I worry that if I stop criticizing myself I will become lazy.",
        "core_beliefs": ["Self-criticism keeps me successful", "Kindness means lowering standards"],
        "target_dynamics": "The patient questions whether self-compassion can coexist with accountability.",
    },
]

# ── 3. 导出 Scenarios 表格 ─────────────────────────────────────────────
scenario_rows = []
for s in SCENARIOS:
    scenario_rows.append({
        "Scenario ID":        s["id"],
        "Category":           s["category"].replace("_", " ").title(),
        "Patient Profile":    s["profile"],
        "Initial Message":    s["initial_patient_message"],
        "Core Beliefs":       " | ".join(s["core_beliefs"]),
        "Target Dynamics":    s["target_dynamics"],
    })
df_scenarios = pd.DataFrame(scenario_rows)

# ── 4. 对话转换函数 ───────────────────────────────────────────────────
def dialogues_to_df(dialogues, model_label):
    rows = []
    for rec in dialogues:
        scenario_id = rec["scenario_id"]
        category    = rec["category"].replace("_", " ").title()
        turn_num    = 0
        for turn in rec["dialogue"]:
            if turn["role"] == "therapist":
                turn_num += 1
            rows.append({
                "Scenario ID":  scenario_id,
                "Category":     category,
                "Model":        model_label,
                "Turn":         turn_num,
                "Speaker":      "Patient" if turn["role"] == "patient" else "Therapist",
                "Content":      turn["content"],
            })
    return pd.DataFrame(rows)

df_base   = dialogues_to_df(base_dialogues,   "Base")
df_ft     = dialogues_to_df(ft_dialogues,     "Fine-tuned")
df_ft_rag = dialogues_to_df(ft_rag_dialogues, "FT+RAG")
df_rag    = dialogues_to_df(rag_dialogues,    "RAG")

# ── 5. 导出 Excel（多 sheet）─────────────────────────────────────────
excel_path = OUT_DIR / "multiturn_dialogues_all_models.xlsx"
writer     = pd.ExcelWriter(excel_path, engine="openpyxl")

# Sheet 1: Scenarios
df_scenarios.to_excel(writer, sheet_name="Scenarios", index=False)

# Sheet 2-5: 四个模型各一个 sheet
df_base.to_excel(  writer, sheet_name="Base",       index=False)
df_ft.to_excel(    writer, sheet_name="Fine-tuned", index=False)
df_ft_rag.to_excel(writer, sheet_name="FT+RAG",     index=False)
df_rag.to_excel(   writer, sheet_name="RAG",        index=False)

# Sheet 6: 所有模型合并（方便对比）
df_all = pd.concat([df_base, df_ft, df_ft_rag, df_rag], ignore_index=True)
df_all.to_excel(writer, sheet_name="All Models Combined", index=False)

# ── 6. 格式美化 ───────────────────────────────────────────────────────
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_COLOR = "2F4A3D"
ALT_COLORS   = {
    "Base":       "F2F2F2",
    "Fine-tuned": "E8F5ED",
    "FT+RAG":     "E3F0FA",
    "RAG":        "F3ECF8",
    "Patient":    "FFFDF9",
    "Therapist":  "F0F4F1",
}

def style_sheet(ws):
    thin = Side(style="thin", color="DDDDDD")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Header row
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", size=10)
        cell.fill      = PatternFill("solid", fgColor=HEADER_COLOR)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = border

    # Data rows
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border    = border
            cell.font      = Font(size=9)

        # Row colour by model or speaker
        model_val   = None
        speaker_val = None
        for cell in row:
            if cell.column == 3:  # Model column
                model_val = str(cell.value) if cell.value else None
            if ws.cell(1, cell.column).value == "Speaker":
                speaker_val = str(cell.value) if cell.value else None

        fill_color = None
        if model_val and model_val in ALT_COLORS:
            fill_color = ALT_COLORS[model_val]
        elif speaker_val and speaker_val in ALT_COLORS:
            fill_color = ALT_COLORS[speaker_val]

        if fill_color:
            for cell in row:
                cell.fill = PatternFill("solid", fgColor=fill_color)

    # Column widths
    col_widths = {}
    for row in ws.iter_rows():
        for cell in row:
            if cell.value:
                col_widths[cell.column] = max(
                    col_widths.get(cell.column, 0),
                    min(len(str(cell.value)), 80)
                )
    for col, width in col_widths.items():
        ws.column_dimensions[get_column_letter(col)].width = max(width * 1.15, 12)

    # Freeze header
    ws.freeze_panes = "A2"

for sheet_name in writer.sheets:
    style_sheet(writer.sheets[sheet_name])

writer.close()
print(f"Saved: {excel_path}")

# ── 7. 同时导出纯文本版（方便直接发给老师）───────────────────────────
txt_path = OUT_DIR / "multiturn_dialogues_all_models.txt"
with open(txt_path, "w", encoding="utf-8") as f:

    # Scenarios
    f.write("=" * 70 + "\n")
    f.write("MULTI-TURN EVALUATION — 12 SCENARIOS\n")
    f.write("=" * 70 + "\n\n")
    for s in SCENARIOS:
        f.write(f"Scenario {s['id']}: {s['category'].replace('_',' ').title()}\n")
        f.write(f"  Profile       : {s['profile']}\n")
        f.write(f"  Core Beliefs  : {' | '.join(s['core_beliefs'])}\n")
        f.write(f"  Initial Msg   : {s['initial_patient_message']}\n")
        f.write(f"  Dynamics      : {s['target_dynamics']}\n\n")

    # Dialogues
    for model_label, dialogues in [
        ("BASE MODEL",       base_dialogues),
        ("FINE-TUNED (FT)",  ft_dialogues),
        ("FT + RAG",         ft_rag_dialogues),
        ("RAG ONLY",         rag_dialogues),
    ]:
        f.write("\n" + "=" * 70 + "\n")
        f.write(f"MODEL: {model_label}\n")
        f.write("=" * 70 + "\n")

        for rec in dialogues:
            sc = next(s for s in SCENARIOS if s["id"] == rec["scenario_id"])
            f.write(f"\n{'─' * 60}\n")
            f.write(f"Scenario {rec['scenario_id']}: {rec['category'].replace('_',' ').title()}\n")
            f.write(f"{'─' * 60}\n")
            turn_num = 0
            for turn in rec["dialogue"]:
                if turn["role"] == "therapist":
                    turn_num += 1
                    speaker = f"[Therapist — Turn {turn_num}]"
                else:
                    speaker = "[Patient]            "
                f.write(f"\n{speaker}\n{turn['content']}\n")
            f.write("\n")

print(f"Saved: {txt_path}")
print(f"\nDone. Files saved to: {OUT_DIR}")
print(f"  - multiturn_dialogues_all_models.xlsx  (Excel, 6 sheets)")
print(f"  - multiturn_dialogues_all_models.txt   (Plain text)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved: /content/drive/MyDrive/CBT/6.25 final results/multiturn_dialogues_all_models.xlsx
Saved: /content/drive/MyDrive/CBT/6.25 final results/multiturn_dialogues_all_models.txt

Done. Files saved to: /content/drive/MyDrive/CBT/6.25 final results
  - multiturn_dialogues_all_models.xlsx  (Excel, 6 sheets)
  - multiturn_dialogues_all_models.txt   (Plain text)
